In [ ]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd
import numpy as np
import seaborn as sns

# Initialize empty lists to store scraped data
Name = []    # List to store product names
Sizes = []   # List to store product sizes
MRPs = []    # List to store product maximum retail prices (MRPs)
Prices = []  # List to store product prices
URLs = []    # List to store product URLs

page = 0  # Initialize page number for pagination

# Loop to iterate through multiple pages of the website
while True:
    
    # Define headers for the HTTP request
    header={
        'Origin': 'https://www.1mg.com',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36'
    }
    
    # Construct the URL for the current page    

    url = "https://www.1mg.com/categories/vitamins-nutrition-5?filter=true&page=" + str(page)
    
    # Send an HTTP GET request to the URL with the defined headers
    res = requests.get(url=url, headers=header)
    
    # Check if the page is not found (404 status)
    if res.status_code == 404:
        print("Page not found. Exiting loop.")
        break
    else:
        # Parse the HTML content of the response using BeautifulSoup
        obj = soup(res.content, 'html.parser')
        
        # Find all product boxes on the current page
        Box = obj.find_all('div', {'class': 'style__product-box___liepi'})
        
        # Loop through each product box and extract relevant information
        for i in Box:
            # Extract product name
            name = i.find('div', {'class': 'style__pro-title___2QwJy'})
            if name:
                Name.append(name.text.strip())
            else:
                Name.append(None)
            
            # Extract product size
            size = i.find('div', {'class': 'style__pack-size___2JQG7'})
            if size:
                Sizes.append(size.text.strip())
            else:
                Sizes.append(None)
            
            # Extract product MRP (maximum retail price)
            mrps = i.find('span', {'class': 'style__discount-price___25Bya'})
            if mrps:
                MRPs.append(mrps.text.replace("₹", ""))
            else:
                MRPs.append(None)
            
            # Extract product price
            price = i.find('div', {'class': 'style__price-tag___cOxYc'})
            if price:
                Prices.append(price.text.replace("₹", ""))
            else:
                Prices.append(None)
            
            # Extract product URL
            mg_url = i.find('a', {'class': 'style__product-link___UB_67'})
            if mg_url:
                b = "https://www.1mg.com" + mg_url.get('href')
                URLs.append(b)
            else:
                URLs.append(None)
        # Print progress information for the current page
        print(page,"page=>",end=" ")
    if page == 251 :
        break
        
    page+=1 # Move to the next page for scraping
    
    print(len(URLs),end=" ")
    print(len(Prices),end=" ")
    print(len(Sizes),end=" ")
    print(len(MRPs),end=" ")
    print(len(Name))

In [ ]:
box=obj.find_all('div',{'class':'style__product-box___liepi'})  #checking the soup object
box

In [ ]:
data={"name":Name,"size_of_bottle":Sizes,"MRPs":MRPs,"selling_price":Prices,"1mg_url":URLs} 
df = pd.DataFrame(data)
df

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import urllib3

# Disable SSL warnings for proxies
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Your working proxies
proxies_list = [
    '89.19.175.122:8008',
    '152.53.168.53:16371',
    '49.229.100.235:8080',
    '152.53.168.53:44985',
    '27.79.213.13:16000',
    '27.79.139.183:16000',
    '223.135.156.183:8080',
    '45.136.198.40:3128',
    '103.169.26.114:8080',
    '152.53.168.53:42086'
]

# Multiple user agents for rotation
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:129.0) Gecko/20100101 Firefox/129.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.6 Safari/605.1.15'
]

def get_random_proxy():
    """Get a random proxy configuration"""
    proxy = random.choice(proxies_list)
    return {
        'http': f'http://{proxy}',
        'https': f'http://{proxy}'
    }

def get_random_headers():
    """Get random headers to avoid detection"""
    return {
        'User-Agent': random.choice(user_agents),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9,hi;q=0.8',
        'Accept-Encoding': 'gzip, deflate, br, zstd',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'none',
        'Sec-Fetch-User': '?1',
        'Cache-Control': 'no-cache',
        'DNT': '1'
    }

def scrape_with_retry(url, max_retries=3):
    """Scrape URL with retry logic and proxy rotation"""
    for attempt in range(max_retries):
        try:
            # Get random proxy and headers
            proxy = get_random_proxy()
            headers = get_random_headers()
            
            # Add referrer sometimes to look more natural
            if random.choice([True, False]):
                headers['Referer'] = 'https://www.1mg.com/'
            
            print(f"  Attempt {attempt + 1}: Using proxy {list(proxy.values())[0].split('//')[1]}")
            
            # Make request with timeout
            res = requests.get(url, headers=headers, proxies=proxy, timeout=20, verify=False)
            
            # Handle different status codes
            if res.status_code == 403:
                print(f"  403 Forbidden - trying different proxy (attempt {attempt + 1})")
                time.sleep(random.uniform(10, 15))
                continue
            elif res.status_code == 429:
                print(f"  Rate limited - waiting longer (attempt {attempt + 1})")
                time.sleep(random.uniform(30, 60))
                continue
            
            res.raise_for_status()
            
            # Parse the content
            soup_obj = BeautifulSoup(res.content, 'html.parser')
            
            # Extract marketer with fallbacks
            marketer_name = None
            
            # Try primary selector
            marketer_tag = soup_obj.find('div', {'class': 'ProductTitle__marketer___7Wsj9'})
            if marketer_tag:
                if marketer_tag.a:
                    marketer_name = marketer_tag.a.text.strip()
                elif marketer_tag.text:
                    marketer_name = marketer_tag.text.strip()
            
            # Try alternative selectors if primary failed
            if not marketer_name:
                alternative_selectors = [
                    ('span', {'class': 'marketer'}),
                    ('div', {'class': 'brand-name'}),
                    ('a', {'class': 'brand-link'}),
                    ('div', {'data-testid': 'brand-name'}),
                    ('span', {'class': 'ProductTitle__brand'}),
                    ('div', {'class': 'manufacturer'})
                ]
                
                for tag, attrs in alternative_selectors:
                    try:
                        element = soup_obj.find(tag, attrs)
                        if element and element.text.strip():
                            marketer_name = element.text.strip()
                            break
                    except:
                        continue
            
            # Clean up the marketer name
            if marketer_name:
                marketer_name = marketer_name.replace('\n', ' ').replace('\t', ' ')
                marketer_name = ' '.join(marketer_name.split())
            
            print(f"  ✓ Success: {marketer_name or 'No marketer found'}")
            return marketer_name
            
        except requests.exceptions.ProxyError as e:
            print(f"  Proxy error (attempt {attempt + 1}): {str(e)[:80]}...")
            time.sleep(random.uniform(3, 8))
            continue
            
        except requests.exceptions.RequestException as e:
            print(f"  Request error (attempt {attempt + 1}): {str(e)[:80]}...")
            time.sleep(random.uniform(2, 5))
            continue
            
        except Exception as e:
            print(f"  Parsing error (attempt {attempt + 1}): {str(e)[:80]}...")
            break
    
    print(f"  ✗ Failed after {max_retries} attempts")
    return None

# Load the Excel file
print("Loading Excel file...")
df = pd.read_excel("C:\\Users\\MICILMEDS\\Documents\\Medi_final\\Homeopathy_Medicines\\Homeopathy_Medicines.xlsx")
df['company'] = None

print(f"Found {len(df)} records to process")
print(f"Using {len(proxies_list)} working proxies")

# Track statistics
success_count = 0
fail_count = 0
start_time = time.time()

# Process each URL
for i in range(len(df)):
    url = df.loc[i, '1mg_url']
    
    print(f"\n[{i+1}/{len(df)}] Processing URL: {url}")
    
    try:
        # Scrape with retry logic
        marketer_name = scrape_with_retry(url)
        df.loc[i, 'company'] = marketer_name
        
        if marketer_name:
            success_count += 1
        else:
            fail_count += 1
            
    except Exception as e:
        print(f"  Unexpected error at index {i}: {e}")
        fail_count += 1
        continue
    
    # Smart delays to avoid detection
    if i % 10 == 0 and i != 0:
        # Save progress every 10 records
        df.to_excel("C:\\Users\\MICILMEDS\\Documents\\Medi_final\\Homeopathy_Medicines\\Homeopathy_Medicines_progress.xlsx", index=False)
        print(f"  Progress saved. Success rate: {success_count/(success_count+fail_count)*100:.1f}%")
    
    if i % 5 == 0 and i != 0:
        # Longer break every 5 records
        delay = random.uniform(8, 15)
        print(f"  Taking {delay:.1f}s break to avoid detection...")
        time.sleep(delay)
    else:
        # Random delay between requests
        delay = random.uniform(3, 7)
        print(f"  Waiting {delay:.1f}s before next request...")
        time.sleep(delay)

# Final save
print("\nSaving final results...")
df.to_excel("C:\\Users\\MICILMEDS\\Documents\\Medi_final\\Homeopathy_Medicines\\Homeopathy_Medicines_updated.xlsx", index=False)

# Print final statistics
end_time = time.time()
total_time = end_time - start_time
total_attempts = success_count + fail_count
success_rate = (success_count / total_attempts * 100) if total_attempts > 0 else 0

print("\n" + "="*60)
print("SCRAPING COMPLETED!")
print(f"Total time: {total_time/3600:.2f} hours")
print(f"Total records processed: {total_attempts}")
print(f"Successful: {success_count}")
print(f"Failed: {fail_count}")
print(f"Success rate: {success_rate:.1f}%")
print(f"Final results saved to: Homeopathy_Medicines_updated.xlsx")
print("="*60)

In [ ]:
df['MRPs'] = df['MRPs'].fillna(df['selling_price'])


In [ ]:
df['MRPs']=df['MRPs'].str.replace('MRP','')
df['selling_price'] = df['selling_price'].str.replace('Discounted Price:','')
df['selling_price']=df['selling_price'].str.replace('MRP','')
df.head(50)

In [ ]:
df['Type of product'] = df['size_of_bottle'].str.split().str[-1]

,name,size_of_bottle,MRPs,selling_price,1mg_url,Type of product
0,Revital Men Multivitamin with Natural Ginseng ...,bottle of 60 soft gelatin capsules,630,593,https://www.1mg.com/otc/revital-men-multivitam...,capsules
1,Enterogermina Probiotic Supplement | For Alter...,strip of 4 capsules,195.79,180,https://www.1mg.com/drugs/enterogermina-probio...,capsules
2,"Centrum Women Vegetarian Tablets for Muscles, ...",bottle of 50 tablets,770,693,https://www.1mg.com/otc/centrum-women-vegetari...,tablets
3,Neurobion Forte Tablet with Vitamin B12 | Help...,strip of 30 tablets,46.1,44.8,https://www.1mg.com/otc/neurobion-forte-tablet...,tablets
4,Protinex High Quality Protein | Nutritional Dr...,jar of 400 gm Powder,675,601,https://www.1mg.com/otc/protinex-high-quality-...,Powder
...,...,...,...,...,...,...
10823,Superkid Kids Multivitamin Gummies (30 Each) S...,combo pack of 3 units,1100,969,https://www.1mg.com/otc/superkid-kids-multivit...,units
10824,Superkid Kids Multivitamin Gummies (30 Each) S...,combo pack of 5 units,1600,1522,https://www.1mg.com/otc/superkid-kids-multivit...,units
10825,Superkid Kids Multivitamin Gummies Strawberry,packet of 4 gummies,99,97.1,https://www.1mg.com/otc/superkid-kids-multivit...,gummies
10826,The Curen Brewing Beauty Collagen Coffee with ...,box of 10 Sachets,1599,1473,https://www.1mg.com/otc/the-curen-brewing-beau...,Sachets


In [ ]:
df.loc[df['Type of product'].str.contains('cr').replace('cr','Strip')]

,name,size_of_bottle,MRPs,selling_price,1mg_url,Type of product
7031,Selemax Capsule CR,strip of 10 capsule cr,250,250,https://www.1mg.com/otc/selemax-capsule-cr-otc...,cr


In [69]:
df1 = df['Type of product'].unique()
df1

array(['capsules', 'tablets', 'Powder', 'Tablets', 'Packs', 'Paste',
       'bottles', 'Jar', 'vegicaps', 'Liquid', 'Syrup', 'Tablet',
       'strips', 'jar', 'Sachets', 'bars', 'Bar', 'gummies', 'Drop',
       'Sachet', 'Granules', 'Solution', 'Suspension', 'sr', 'servings',
       'Oil', 'md', 'Bottle', 'er', 'Unit', 'Drops', 'Juice', 'Seeds',
       'boxes', 'Tonic', 'Pack', 'tablet', 'Cream', 'vials', 'Tubes',
       'Kit', 'Serum', 'DR', 'Butter', 'Spray', 'Shampoo', 'Fruits',
       'Gargle', 'Leaves', 'Strip', 'Snacks', 'units', 'soflets',
       'Injection', 'solutions', 'Wash', 'Box', 'Lotion', 'Cookie', 'dt',
       'caplets', 'Cereal', 'Tincture', 'straws', 'jelly', 'Gel',
       'Capsule', 'lozenges', 'Gummy', 'Berry', 'Elixir', 'cookies', 'cr',
       'powders', 'Muesli', 'Tea', 'tabcaps', 'Tube', 'Cleanser',
       'Moisturiser', 'Resin', 'Vegicap', 'Jelly', 'Beans'], dtype=object)

In [ ]:

df.to_csv('C:\\Users\\MICILMEDS\\Documents\\Medi_final\\correct_data\\vitamins-nutrition.csv', index=False)